In [52]:
import pandas as pd





index_df  = pd.read_csv("./screens_biogrid/BIOGRID-ORCS-SCREEN_INDEX-2.0.18.index.tab.txt", sep="\t")
print(type(index_df))








<class 'pandas.core.frame.DataFrame'>


In [53]:
index_df.columns

Index(['#SCREEN_ID', 'SOURCE_ID', 'SOURCE_TYPE', 'AUTHOR', 'SCREEN_NAME',
       'SCORES_SIZE', 'FULL_SIZE', 'FULL_SIZE_AVAILABLE', 'NUMBER_OF_HITS',
       'ANALYSIS', 'SIGNIFICANCE_INDICATOR', 'SIGNIFICANCE_CRITERIA',
       'THROUGHPUT', 'SCREEN_TYPE', 'SCREEN_FORMAT', 'EXPERIMENTAL_SETUP',
       'DURATION', 'CONDITION_NAME', 'CONDITION_DOSAGE', 'MOI', 'LIBRARY',
       'LIBRARY_TYPE', 'LIBRARY_METHODOLOGY', 'ENZYME', 'CELL_LINE',
       'CELL_TYPE', 'PHENOTYPE', 'SCORE_COL_COUNT', 'SCORE.1_TYPE',
       'SCORE.2_TYPE', 'SCORE.3_TYPE', 'SCORE.4_TYPE', 'SCORE.5_TYPE',
       'ORGANISM_ID', 'ORGANISM_OFFICIAL', 'NOTES', 'SOURCE',
       'SCREEN_RATIONALE'],
      dtype='object')

In [54]:
index_df[index_df['#SCREEN_ID'].isin([5, 6])]



,#SCREEN_ID,SOURCE_ID,SOURCE_TYPE,AUTHOR,SCREEN_NAME,SCORES_SIZE,FULL_SIZE,FULL_SIZE_AVAILABLE,NUMBER_OF_HITS,ANALYSIS,...,SCORE.1_TYPE,SCORE.2_TYPE,SCORE.3_TYPE,SCORE.4_TYPE,SCORE.5_TYPE,ORGANISM_ID,ORGANISM_OFFICIAL,NOTES,SOURCE,SCREEN_RATIONALE
2,5,25307932,pubmed,Gilbert LA (2014),2-PMID25307932,15976,15976,Yes,1345,Log2 Fold Change (L2FC),...,Gamma (normalized log2e/t),-,-,-,-,9606,H. sapiens,inhibition of hit genes results in decreased f...,BioGRID ORCS,Gene inhibition decreases fitness
3,6,25307932,pubmed,Gilbert LA (2014),3-PMID25307932,15977,15977,Yes,142,Log2 Fold Change (L2FC),...,Gamma (normalized log2e/t),-,-,-,-,9606,H. sapiens,activation of hit genes results in decreased f...,BioGRID ORCS,Gene activation decreases fitness


In [55]:
# Step 1: Only essentiality screens (not drug resistance, viral, etc.)
essential = index_df[index_df["SCREEN_RATIONALE"].str.contains("essential", case=False, na=False)]



# Step 2: Only cell proliferation phenotype
essential = essential[essential["PHENOTYPE"].str.contains("proliferation", case=False, na=False)]


# Step 3: No drug condition (pure essentiality, no confounding)
essential = essential[essential["CONDITION_NAME"] == "-"]
essential = essential[essential["SCORE.1_TYPE"] == "Bayes Factor"]

# Step 4: Split cancer vs normal by CELL_TYPE
cancer_ids = essential[essential["CELL_TYPE"].str.contains("Cancer|Leukemia|Carcinoma|Lymphoma|Melanoma|Glioma|Sarcoma", case=False, na=False)]["#SCREEN_ID"]

normal_ids = essential[~essential["CELL_TYPE"].str.contains("Cancer|Leukemia|Carcinoma|Lymphoma|Melanoma|Glioma|Sarcoma", case=False, na=False)]["#SCREEN_ID"]

print(f"Cancer essentiality screens: {len(cancer_ids)}")

print(f"Normal essentiality screens: {len(normal_ids)}")

print(normal_ids)

Cancer essentiality screens: 275
Normal essentiality screens: 87
10        27
11        28
452      570
453      571
454      572
        ... 
1446    1858
1447    1859
1448    1860
1608    2062
1609    2063
Name: #SCREEN_ID, Length: 87, dtype: int64


In [56]:
file_list_cancer = []
file_list_normal = []
for screen_id in cancer_ids:
    file_list_cancer.append(f"./screens_biogrid/BIOGRID-ORCS-SCREEN_{screen_id}-2.0.18.screen.tab.txt")

for screen_id in normal_ids:
    file_list_normal.append(f"./screens_biogrid/BIOGRID-ORCS-SCREEN_{screen_id}-2.0.18.screen.tab.txt")




In [57]:
screen = pd.read_csv(file_list_cancer[6], sep="\t")
screen.head()

,#SCREEN_ID,IDENTIFIER_ID,IDENTIFIER_TYPE,OFFICIAL_SYMBOL,ALIASES,ORGANISM_ID,ORGANISM_OFFICIAL,SCORE.1,SCORE.2,SCORE.3,SCORE.4,SCORE.5,HIT,SOURCE
0,691,6222,ENTREZ_GENE,RPS18,D6S218E|HKE3|KE-3|KE3|S18,9606,Homo sapiens,3.738608,-,-,-,-,YES,BioGRID ORCS
1,691,6208,ENTREZ_GENE,RPS14,EMTB|S14,9606,Homo sapiens,3.738608,-,-,-,-,YES,BioGRID ORCS
2,691,6132,ENTREZ_GENE,RPL8,L8,9606,Homo sapiens,3.738608,-,-,-,-,YES,BioGRID ORCS
3,691,6168,ENTREZ_GENE,RPL37A,L37A,9606,Homo sapiens,3.738608,-,-,-,-,YES,BioGRID ORCS
4,691,25873,ENTREZ_GENE,RPL36,L36|DKFZp566B023,9606,Homo sapiens,3.738608,-,-,-,-,YES,BioGRID ORCS


In [106]:
class Gene:
    def __init__(self, identifier_id, symbol=None, total_screens=1):
        self.identifier_id = identifier_id
        self.symbol = symbol
        self.scores = []
        self.repetitions = []
        self.hit = []
        self.total_screens = total_screens
        




    def add_score(self, score):
        self.scores.append(score)

    def add_repetition(self, repetition):
        self.repetitions.append(repetition)

    def add_hit(self, hit):
        self.hit.append(hit)

    def sum_scores(self):  
        return sum(self.scores)
  
    def num_repetitions(self):
        return len(self.repetitions)
    
    def avg_occurance(self):
        if self.repetitions:
            return len(self.repetitions) / self.total_screens
        return 0
    def average_score(self):
        if self.scores:
            return self.sum_scores() / len(self.scores)
        return 0
    def combined_score(self):
        return self.avg_occurance() * self.average_score()





def process_screens(file_list, threshold):
    gene_map = {}
    error_screens = []
    row_error_count = 0
    total_row_count = 0
    total_screens = len(file_list)
    for file in file_list:
        try:
            screen = pd.read_csv(file, sep="\t")
            for index,row in screen.iterrows():
                total_row_count += 1
                try:
                    int(row['IDENTIFIER_ID'])  # Check if IDENTIFIER_ID is an integer
                    float(row['SCORE.1'])  # Check if SCORE.1 is a float
                    if row['IDENTIFIER_ID'] not in gene_map and row['SCORE.1'] >= threshold:
                        gene_map[row['IDENTIFIER_ID']] = Gene(row['IDENTIFIER_ID'], symbol=row['OFFICIAL_SYMBOL'], total_screens=total_screens)

                    if row['SCORE.1'] >= threshold:
                        gene_map[row['IDENTIFIER_ID']].add_score(row['SCORE.1'])
                        gene_map[row['IDENTIFIER_ID']].add_repetition(row['#SCREEN_ID'])
                        gene_map[row['IDENTIFIER_ID']].add_hit(row['HIT'])
                except Exception as e:
                    row_error_count += 1
        except Exception as e:
            error_screens.append(file)
            print(f"Error processing screen {file} : {e}")

    return gene_map, error_screens, row_error_count, total_row_count, total_screens


In [ ]:
gene_map = {}
g1 = Gene(1, 'gene1', 5)
g1.add_score(10)
g1.add_score(20)
g1.add_repetition('s1')
g1.add_repetition('s2')
g2 = Gene(2, 'gene2', 5)

gene_map[1] = g1

gene_map[2] = g2
print(gene_map[2].scores)




[10, 20]


In [108]:
gene_map_cancer, error_screens_cancer, row_error_count_cancer, total_row_count_cancer, total_screens_cancer = process_screens(file_list_cancer, 5)
gene_map_normal, error_screens_normal, row_error_count_normal, total_row_count_normal, total_screens_normal = process_screens(file_list_normal, 5)

In [115]:
print(row_error_count_cancer)

992


In [ ]:
normal_gene_ids = {
    gid for gid, gene in gene_map_normal.items()
    
}

# creating cancer gene list which is not present in essential gene list for normal cell
cancer_specific_gene = [
    gene for gid, gene in gene_map_cancer.items()
    if gid not in normal_gene_ids
]

# Step 3: sort by average score, descending, take top 10 gene to knowck out with combined score
top5 = sorted(cancer_specific_gene, key=lambda g: g.combined_score(), reverse=True)[:10]

for g in top5:
    print(g.symbol, g.identifier_id, g.combined_score())


# Step 4: sort by average score, descending, take top 10 gene to knowck out with average score
top5 = sorted(cancer_specific_gene, key=lambda g: g.average_score(), reverse=True)[:10]

for g in top5:
    print(g.symbol, g.identifier_id, g.average_score())


GART 2618 2.4652145454545455
WRN 7486 1.4221127272727274
TFRC 7037 1.1656181818181819
FH 2271 0.8016145454545456
TIMM17A 10440 0.74392
ALDH18A1 5832 0.7281303030302909
CTNNB1 1499 0.6149163636363637
CRTC3 64784 0.5674327272727273
LIPT2 387787 0.5459471515151636
JUNB 3726 0.4250945454545454
CRTC3 64784 156.044
WRN 7486 130.36033333333333
GART 2618 112.98899999999999
TIMM17A 10440 102.289
TPK1 27010 99.27
TRIM8 81603 90.84
TTC9C 283237 90.096
BCAS4 55653 69.008
CTNNB1 1499 56.367333333333335
ZBTB8OS 339487 55.584


In [112]:
# Load BAGEL CEGv2 reference list
depmap_essentials = pd.read_csv("./screens_biogrid/CRISPRInferredCommonEssentials.csv")
print(depmap_essentials.columns)
depmap_essentials.head()

import re

def extract_entrez_id(gene_str):
    match = re.search(r"\((\d+)\)", gene_str)
    return int(match.group(1)) if match else None

depmap_essentials["ENTREZ_ID"] = depmap_essentials.iloc[:, 0].apply(extract_entrez_id)
depmap_ids = set(depmap_essentials["ENTREZ_ID"].dropna().astype(int))


cancer_specific_gene_refined = [
    gene for  gene in cancer_specific_gene
    if  gene.identifier_id not in depmap_ids
]



print(f"After DepMap filter: {len(cancer_specific_gene_refined)}")
print("combined_score")
# Step 3: sort by average score, descending, take top 10 gene to knowck out with combined score
top5 = sorted(cancer_specific_gene_refined, key=lambda g: g.combined_score(), reverse=True)[:10]

for g in top5:
    print(g.symbol, g.identifier_id, g.combined_score())

print("Average")

# Step 4: sort by average score, descending, take top 10 gene to knowck out with average score
top5 = sorted(cancer_specific_gene_refined, key=lambda g: g.average_score(), reverse=True)[:10]

for g in top5:
    print(g.symbol, g.identifier_id, g.average_score())




Index(['Essentials'], dtype='object')
After DepMap filter: 1531
combined_score
GART 2618 2.4652145454545455
WRN 7486 1.4221127272727274
TFRC 7037 1.1656181818181819
FH 2271 0.8016145454545456
TIMM17A 10440 0.74392
ALDH18A1 5832 0.7281303030302909
CTNNB1 1499 0.6149163636363637
CRTC3 64784 0.5674327272727273
LIPT2 387787 0.5459471515151636
JUNB 3726 0.4250945454545454
Average
CRTC3 64784 156.044
WRN 7486 130.36033333333333
GART 2618 112.98899999999999
TIMM17A 10440 102.289
TPK1 27010 99.27
TRIM8 81603 90.84
TTC9C 283237 90.096
BCAS4 55653 69.008
CTNNB1 1499 56.367333333333335
RPS26 6231 55.556


In [113]:
import re

# Common housekeeping/pan-essential gene symbol prefixes
housekeeping_pattern = re.compile(
    r"^("
    r"RPL\d|RPS\d|MRPL\d|MRPS\d"      # ribosomal proteins (cytoplasmic + mitochondrial)
    r"|EIF\d|EEF\d"                    # translation initiation/elongation factors
    r"|POLR\d|POLE\d?|POLD\d?"         # RNA/DNA polymerase subunits
    r"|PSMA\d|PSMB\d|PSMC\d|PSMD\d"    # proteasome subunits
    r"|SNRNP\d|SNRPA|SNRPB|SNRPD|SF3[AB]" # spliceosome components
    r"|[A-Z]{1,3}ARS\d?$"              # aminoacyl-tRNA synthetases (e.g. YARS, AARS)
    r"|COX\d|NDUF|ATP5"                # mitochondrial respiratory chain
    r"|HIST\d|H2A|H2B|H3F|H4C"         # core histones
    r")"
)

known_pan_essential_symbols = {
    "PRPF19", "PCNA", "KIF11", "DCAF1",  # add as you discover more
}

def is_housekeeping_symbol(symbol):
    if not symbol:
        return False
    if symbol in known_pan_essential_symbols:
        return True
    return bool(housekeeping_pattern.match(symbol))

cancer_specific_gene_refined_2 = [
    gene for  gene in cancer_specific_gene_refined
    if  not is_housekeeping_symbol(gene.symbol)
]



print(f"After DepMap filter: {len(cancer_specific_gene_refined_2)}")
print("combined")
# Step 3: sort by average score, descending, take top 10 gene to knowck out with combined score
top5 = sorted(cancer_specific_gene_refined_2, key=lambda g: g.combined_score(), reverse=True)[:10]

for g in top5:
    print(g.symbol, g.identifier_id, g.combined_score())

print("average")
# Step 4: sort by average score, descending, take top 10 gene to knowck out with average score
top5 = sorted(cancer_specific_gene_refined_2, key=lambda g: g.average_score(), reverse=True)[:10]

for g in top5:
    print(g.symbol, g.identifier_id, g.average_score())


After DepMap filter: 1441
combined
GART 2618 2.4652145454545455
WRN 7486 1.4221127272727274
TFRC 7037 1.1656181818181819
FH 2271 0.8016145454545456
TIMM17A 10440 0.74392
ALDH18A1 5832 0.7281303030302909
CTNNB1 1499 0.6149163636363637
CRTC3 64784 0.5674327272727273
LIPT2 387787 0.5459471515151636
JUNB 3726 0.4250945454545454
average
CRTC3 64784 156.044
WRN 7486 130.36033333333333
GART 2618 112.98899999999999
TIMM17A 10440 102.289
TPK1 27010 99.27
TRIM8 81603 90.84
TTC9C 283237 90.096
BCAS4 55653 69.008
CTNNB1 1499 56.367333333333335
PMVK 10654 55.539500000000004


In [ ]:
lis = [7157,3845,1956,4609,5290,5728,5925,672,675,324,7428,1029,4893,673,2064,4233,238,3717,3417,1499]
for i in lis:
    if i in gene_map_cancer:
        g = gene_map_cancer[i]
        print(f"Gene: {g.symbol}, ID: {g.identifier_id}, Avg Score: {g.average_score()}, Combined Score: {g.combined_score()}")
    else:
        print(f"Gene ID {i} not found in cancer gene map.")


Gene: TP53, ID: 7157, Avg Score: -80.018, Combined Score: -0.5819490909090909
Gene: KRAS, ID: 3845, Avg Score: -65.6055, Combined Score: -0.47713090909090916
Gene: EGFR, ID: 1956, Avg Score: -64.664, Combined Score: -0.47028363636363635
Gene: MYC, ID: 4609, Avg Score: 54.8275, Combined Score: 0.39874545454545457
Gene: PIK3CA, ID: 5290, Avg Score: -34.915, Combined Score: -0.2539272727272727
Gene: PTEN, ID: 5728, Avg Score: -74.361, Combined Score: -0.5408072727272728
Gene: RB1, ID: 5925, Avg Score: -81.53200000000001, Combined Score: -0.59296
Gene: BRCA1, ID: 672, Avg Score: 41.432, Combined Score: 0.3013236363636364
Gene: BRCA2, ID: 675, Avg Score: 40.61, Combined Score: 0.2953454545454545
Gene: APC, ID: 324, Avg Score: -54.073, Combined Score: -0.39325818181818184
Gene: VHL, ID: 7428, Avg Score: 49.995000000000005, Combined Score: 0.36360000000000003
Gene: CDKN2A, ID: 1029, Avg Score: -87.737, Combined Score: -0.6380872727272727
Gene: NRAS, ID: 4893, Avg Score: -69.37299999999999, Co

In [93]:
gene_map_cancer[673].repetitions

[1458, 1459]